# Training & Evaluation

Train both BiLSTM and Transformer models using optimized hyperparameters from `01_optimization.ipynb`.

In [ ]:
import sys; sys.path.insert(0, '..')
from src import suppress_logs; suppress_logs()

import torch
torch.set_float32_matmul_precision('medium')

import json
from pathlib import Path
from pytorch_lightning import seed_everything
from src.data import GestureDataModule
from src.models import BiLSTMModule, TransformerModule
from src.training import CrossValidator
from src.visualization import plot_cv_comparison, print_results_table

## Configuration

In [ ]:
EPOCHS = 50
PATIENCE = 15
N_FOLDS = 5
SEED = 42
DATA_PATH = None  # Auto-download from HuggingFace

seed_everything(SEED)
print(f'Training: {EPOCHS} epochs, {N_FOLDS}-fold CV')

## Load Optimized Hyperparameters

Loads best parameters from `01_optimization.ipynb` output. Falls back to defaults if not found.

In [ ]:
OPTUNA_DIR = Path('../results/optuna')

# Default hyperparameters (fallback)
DEFAULT_PARAMS = {
    'bilstm': {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.15, 'learning_rate': 0.002, 'optimizer': 'nadam'},
    'transformer': {'d_model': 64, 'nhead': 8, 'num_layers': 2, 'dim_feedforward': 128, 'dropout': 0.1, 'learning_rate': 0.001}
}

def load_best_params(model_name):
    """Load best params from JSON or use defaults."""
    json_path = OPTUNA_DIR / f'{model_name}_best_params.json'
    if json_path.exists():
        with open(json_path) as f:
            data = json.load(f)
            print(f'{model_name.upper()}: Loaded from {json_path} (acc={data["best_accuracy"]:.4f})')
            return data['params']
    else:
        print(f'{model_name.upper()}: Using defaults (run in 01_optimization.ipynb first)')
        return DEFAULT_PARAMS[model_name]

BILSTM_PARAMS = load_best_params('bilstm')
TRANSFORMER_PARAMS = load_best_params('transformer')

print('\nBiLSTM params:', BILSTM_PARAMS)
print('Transformer params:', TRANSFORMER_PARAMS)

## Load Data

In [ ]:
dm = GestureDataModule(data_path=DATA_PATH, seed=SEED)
dm.setup()
print(f'Classes: {dm.class_names}')
print(f'Input size: {dm.input_size}, Samples: {len(dm.train_dataset) + len(dm.val_dataset)}')

## Train Both Models

In [ ]:
results = {}

# BiLSTM
print('\n' + '='*60 + '\nTraining BiLSTM\n' + '='*60)
cv_bilstm = CrossValidator(BiLSTMModule, dm, N_FOLDS, {'max_epochs': EPOCHS, 'accelerator': 'auto'})
results['bilstm'] = cv_bilstm.run(BILSTM_PARAMS, patience=PATIENCE)

# Transformer
print('\n' + '='*60 + '\nTraining Transformer\n' + '='*60)
cv_transformer = CrossValidator(TransformerModule, dm, N_FOLDS, {'max_epochs': EPOCHS, 'accelerator': 'auto'})
results['transformer'] = cv_transformer.run(TRANSFORMER_PARAMS, patience=PATIENCE)

## Results Comparison

In [ ]:
print_results_table(results)

In [ ]:
fig = plot_cv_comparison(results, 'Cross-Validation Results')
Path('../plots').mkdir(exist_ok=True)
fig.savefig('../plots/cv_comparison.png', dpi=150)

## Save Results

In [ ]:
Path('../results/training').mkdir(parents=True, exist_ok=True)
for name, r in results.items():
    with open(f'../results/training/{name}_cv_results.json', 'w') as f:
        json.dump(r.to_dict(), f, indent=2)
print('Results saved to ../results/training/')